# Model Comparison

Let's see how well the models went and choose the best one.

In [0]:
%pip install feature_engine
%pip install xgboost lightgbm catboost
%pip install seaborn

In [0]:
import pandas as pd
import mlflow

In [0]:
# experiment_id = 1316272881415465
experiment_id = 1923915617965994

max_results = 1000  # Maximum number of rows to retrieve
runs = mlflow.search_runs(experiment_ids=[experiment_id], max_results=max_results)

# Display dataframe in cell output
display(runs)


In [0]:
import mlflow
import pandas as pd

# 1. Search all runs — order by a metric that exists on BOTH parent and child runs,
#    or just don't order (leaderboard sorts in pandas anyway)
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["attributes.start_time DESC"],
)

# 2. Drop child runs created by GridSearchCV autologging
if "tags.mlflow.parentRunId" in runs.columns:
    runs = runs[runs["tags.mlflow.parentRunId"].isna()].copy()

# 3. Keep only runs that have your primary evaluation metric
PRIMARY_METRIC = "metrics.test_r2"   # change to metrics.auc_test if classification
if PRIMARY_METRIC in runs.columns:
    runs = runs[runs[PRIMARY_METRIC].notna()].copy()

# 4. Optional: restrict to FINISHED runs only
runs = runs[runs["status"] == "FINISHED"].copy()

# 5. Column map (include params AND metrics you care about)
columns_map = {
    "tags.mlflow.runName":               "Model",
    "tags.estimator_name":               "Estimator",
    "metrics.best_cv_score":             "CV Score",
    "metrics.train_r2":                  "Train R²",
    "metrics.test_r2":                   "Test R²",
    "metrics.train_rmse":                "Train RMSE",
    "metrics.test_rmse":                 "Test RMSE",
    "metrics.train_mae":                 "Train MAE",
    "metrics.test_mae":                  "Test MAE",
    "metrics.train_mape":                "Train MAPE",
    "metrics.test_mape":                 "Test MAPE",
    "params.alpha":                      "alpha",
    "params.best_alpha":                 "best_alpha",
    "params.model_type":                 "model_type",
    "start_time":                        "Started",
    "end_time":                          "Ended",
    "run_id":                            "Run ID",
}

available = [c for c in columns_map if c in runs.columns]
lb = runs[available].rename(columns=columns_map)

# 6. Sort by primary metric, highest first
if "Test R²" in lb.columns:
    lb = lb.sort_values("Test R²", ascending=False, na_position="last").reset_index(drop=True)

# 7. Round floats
float_cols = lb.select_dtypes(include=["float64", "float32"]).columns
lb[float_cols] = lb[float_cols].round(4)

# 8. Drop duplicates BEFORE display (your original call was .drop_duplicates on display, after sort)
lb = lb.drop_duplicates(subset=["Run ID"] if "Run ID" in lb.columns else None)

display(lb.drop_duplicates())

In [0]:
lb = lb[
    lb["Estimator"].notna()
    & ~lb["Estimator"].astype(str).str.strip().str.lower().isin(["", "nan", "none"])
]

display(lb.drop_duplicates().reset_index(drop=True))

# Champion Model

In [0]:
import mlflow

# Import Model
# model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_baseline_ols/1")
# model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_model/8")
model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_tuned/2")

## Feature Importance


In [0]:

from sklearn.model_selection import train_test_split

# Table
tb_abt = 'raw_abt1_totalcharges'

df = spark.table(f'workspace.default.{tb_abt}').toPandas()

X = df.drop(columns=['customerID', 'TotalCharges']).fillna(0).astype(float)
y = df["TotalCharges"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Size Train dataset: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Target mean: ({y_train.mean():.2f})')

print(f'Size Test dataset: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Target mean: ({y_test.mean():.2f})')

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Get the regressor and feature names
# Case A: If it's a Pipeline (e.g. Pipeline([('scaler', StandardScaler()), ('regressor', LinearRegression())]))
if hasattr(model, "named_steps"):
    # Grab the last step (the regressor)
    regressor = model.named_steps[list(model.named_steps.keys())[-1]]
    # Get feature names from the pipeline if available, or pass your X_train.columns
    if hasattr(model, "feature_names_in_"):
        feature_names = model.feature_names_in_
    else:
        feature_names = X_train.columns  # fallback to your training dataframe columns
# Case B: If it's a standalone model (LinearRegression or Ridge directly)
else:
    regressor = model
    feature_names = getattr(model, "feature_names_in_", X_train.columns)


# -- Getting Feature and Coefficients
# Create a clean DataFrame of Coefficients
df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': regressor.coef_,
    'Abs_Importance': abs(regressor.coef_)
}).sort_values(by='Abs_Importance', ascending=False).reset_index(drop=True)

# Display the table in Databricks
display(df_importance)

# Plot the top features
plt.figure(figsize=(10, 8))
df_importance.head(15).sort_values(by='Abs_Importance', ascending=True).plot(
    x='Feature', 
    y='Coefficient', 
    kind='barh', 
    legend=False, 
    color='steelblue',
    edgecolor='black'
)
plt.title("Top Feature Coefficients (Linear Model)", fontsize=14)
plt.xlabel("Coefficient Value")
plt.axvline(0, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

# --- Get Feature Importance
# Calculate Real Impact (|coef| * standard deviation)
std_devs = X_train[feature_names].std().values
real_impact = np.abs(regressor.coef_ * std_devs)

# Convert to Percentage (%)
importance_pct = (real_impact / real_impact.sum()) * 100

df_pct = pd.DataFrame({
    'Feature': feature_names,
    'Importance (%)': np.round(importance_pct, 2)
}).sort_values(by='Importance (%)', ascending=False).reset_index(drop=True)

# Display Table
display(df_pct)

# Plot Percentage Bar Chart
plt.figure(figsize=(9, 4))
bars = plt.barh(df_pct['Feature'][::-1], df_pct['Importance (%)'][::-1], color='#2b5c8f', edgecolor='black')

# Add % labels on each bar
plt.bar_label(bars, fmt='%.2f%%', padding=5, fontsize=10, fontweight='bold')

plt.title("Feature Importance by % (TotalCharges Imputer)", fontsize=13, pad=15)
plt.xlabel("Relative Importance (%)", fontsize=11)
plt.xlim(0, 110)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Why is this my champion model?

This is an imputer, not at a model, so having it at 99% is not a bad thing.
Actually it says that the relationship is preserved: `TotalCharges ≈ tenure * MonthlyCharges (TotalSpend)`

### This is an Imputer, Not the Churn Model
- If this were our final churn prediction model and one variable had 99% importance, that would be a red flag (it would mean you have target leakage, like using CustomerFeedback where the customer says "I canceled").
- But this is an Imputation Model specifically built to reconstruct a physical billing column: `TotalCharges ≈ tenure × MonthlyCharges = TotalSpend`
- In business billing, cumulative spend should equal monthly fee multiplied by months. The fact that the model relies 99% on TotalSpend proves that:
    1. The physics of the billing data are respected.
    2. The model didn't latch onto weird correlations (like customer gender or whether they stream movies).

### What the Remaining 1% Represents
- The remaining ~0.1% to 1% is the small real-world noise:
    - Price changes over time (e.g. someone had a promotional rate in month 1–3).
    - Taxes, fees, or contract discounts (isContractTwoYear).

The model is doing what it was designed to do: it takes TotalSpend as the foundation (99%) and applies minor adjustments (1%) for contractual discounts.


Once we have a champion model, we regiter this model.
We are using autolog, so all models are being registered now. Easier for a Hackatohn. In reallife it can get clutter fast, so no need to use the autolog. (future reference).

To know which models we are working on we set "Alias"to champion, so now we dont have to remember wich version is the best we already eye ball the models and choose the best one. Also avoid  error and tests runs (when using autlog) if we get the latest only.



# Set Champion Model

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

# Set alias 'champion' to Version 2 of your tuned model
client.set_registered_model_alias(
    name="workspace.default.total_charges_ridge_tuned",
    alias="champion",
    version="2"  
)

print("Champion alias successfully set!")

In [0]:
# Champion model
# model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_tuned/latest")
model = mlflow.sklearn.load_model("models:/workspace.default.total_charges_ridge_tuned@champion")

model